In [ ]:
import os
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    RunResult,
    OpenAIChatCompletionsModel,
    ModelSettings,
    function_tool,
)
from agents.tracing import set_tracing_disabled


In [20]:
local_model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    )
)

In [21]:
agent = Agent(
    name="Greeter",
    instructions="You are a welcoming and friendly assistant. Be concise.",
    model=local_model
)

In [22]:
result = await Runner.run(agent, "What is the capital of Pakistan?")

print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


The capital of Pakistan is **Islamabad**.


OPENAI_API_KEY is not set, skipping trace export


In [25]:
agent = Agent(
    name="Analyst", 
    instructions="You analyze topics briefly.",
    model=local_model
)

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

OPENAI_API_KEY is not set, skipping trace export


Final Output:  

**Python is the go‑to language for most AI/ML work today.**  
Its popularity stems from a combinat...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


OPENAI_API_KEY is not set, skipping trace export


In [26]:
agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Turn 1: 

A **list** in Python is an ordered, mutable collection of items.

**Key characteristics:**
- Ordered (items have a defined index)
- Mutable (you can add, remove, or modify items)
- Can hold mixed data types

**Basic operations:**

```python
# Creating a list
my_list = [1, 2, 3]

# Adding items
my_list.append(4)        # [1, 2, 3, 4]
my_list.insert(0, 0)     # [0, 1, 2, 3, 4]

# Accessing items
my_list[0]               # 0
my_list[-1]             # 4 (last item)

# Slicing
my_list[1:3]            # [1, 2]

# Removing items
my_list.remove(0)       # removes first occurrence of 0
popped = my_list.pop()  # removes and returns last item
```

Lists are one of Python's most commonly used data structures.



OPENAI_API_KEY is not set, skipping trace export


Turn 2: 

Two main ways to sort a list in Python:

**1. `sort()` method** — sorts in place (modifies original list)

```python
nums = [3, 1, 4, 1, 5]
nums.sort()
print(nums)  # [1, 1, 3, 4, 5]
```

**2. `sorted()` function** — returns a new list (original unchanged)

```python
nums = [3, 1, 4, 1, 5]
sorted_nums = sorted(nums)
print(sorted_nums)  # [1, 1, 3, 4, 5]
print(nums)         # [3, 1, 4, 1, 5] (unchanged)
```

**Reverse sorting:**

```python
nums.sort(reverse=True)
nums.sort()          # then reverse
```

**Sort with custom key:**

```python
# Sort by length of strings
words = ["apple", "pie", "hi"]
words.sort(key=len)  # ["hi", "pie", "apple"]
```

Use `sort()` when you don't need the original list; use `sorted()` when you want to keep it intact.



OPENAI_API_KEY is not set, skipping trace export


Turn 3: 

Two ways to reverse sort:

**1. Using `reverse=True` parameter:**

```python
nums = [3, 1, 4, 1, 5]
nums.sort(reverse=True)
# [5, 4, 3, 1, 1]
```

**2. Using `reverse()` method** (reverses order, not sorts):

```python
nums = [3, 1, 4, 1, 5]
nums.reverse()
# [5, 1, 4, 1, 3]
```

**With `sorted()`:**

```python
nums = [3, 1, 4, 1, 5]
sorted(nums, reverse=True)  # [5, 4, 3, 1, 1]
```

| Method | What it does |
|--------|--------------|
| `sort(reverse=True)` | Sorts descending |
| `sorted(x, reverse=True)` | Sorts descending, new list |
| `reverse()` | Just flips the order |

Note: `reverse()` doesn't sort—it only reverses the current order.


OPENAI_API_KEY is not set, skipping trace export


In [28]:
agent = Agent(
    name="Streamer", 
    instructions="Explain briefly.",
    model=local_model
)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

The user wants a concise explanation of machine learning in exactly 2 sentences. I need to define what it is in a clear, accessible way.Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. It works by algorithms that improve their performance on a task as they are exposed to more examples over time.

--- Final: Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. It works by algorithms that improve their performance on a task as they are exposed to more examples over time.


OPENAI_API_KEY is not set, skipping trace export


In [29]:
agent = Agent(
    name="Lahore Guide",
    instructions="You are Lahore Guide, a smart, friendly, and highly knowledgeable local assistant for Lahore, Pakistan. You help users explore the city like a local",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "Suggest places to visit (historical, food, shopping, entertainment?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "Can you narrow it down to places in Gulberg and DHA, and keep it budget-friendly?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "Great, now make a half-day plan from these options with minimum travel time and include food stops."}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

OPENAI_API_KEY is not set, skipping trace export


Turn 1: # 🏛️ Explore Lahore: Complete Visitor Guide

## 📜 Historical & Heritage Sites

**Must-Visit:**
- **Badshahi Mosque** – Mughal-era grandeur, one of the largest mosques in the world
- **Lahore Fort (Shahi Qila)** – UNESCO World Heritage Site with stunning architecture
- **Minar-e-Pakistan** – Iconic landmark commemorating the Lahore Resolution
- **Wazir Khan Mosque** – A hidden gem with stunning fresco artwork
- **Shalimar Gardens** – Another UNESCO site, perfect for Mughal garden lovers
- **Wazir Khan Hammam** – Restored 17th-century bathhouse with beautiful tilework
- **Allama Iqbal's Tomb** – Final resting place of Pakistan's national poet
- **Tomb of Emperor Jahangir** – Beautiful Mughal-era monument

---

## 🍜 Food Spots (Street Food to Fine Dining)

**Street Food Legends:**
- **Gulshan Sweets** – Best *jalebi* and *mithai* in town
- **Dhabba** (Gawalmandi) – Authentic Punjabi *daal*, *tandoori rotis*
- **Mandelay** – Famous *samosas* and *chaat*
- **Baba Bulleh Shah Tikka K

OPENAI_API_KEY is not set, skipping trace export


Turn 2: # 🏙️ Budget-Friendly Guide: Gulberg & DHA, Lahore

---

## ⚠️ Note
Gulberg and DHA are **modern, commercial areas** — not much historical heritage here, but great for **food, shopping, and casual hangouts**!

---

## 🍜 Budget Food Spots

### Gulberg (Affordable Picks)
| Place | What's Good | Price Range |
|-------|-------------|-------------|
| **Hasty Tasty** | Pakistani desi food, cheap & filling | ₹₵ |
| **Khatri Food Corner** | Classic Pakistani breakfast & snacks | ₹₵ |
| **Al-Madina Sandwich Shop** | Budget sandwiches & rolls | ₹ |
| **Tikka Shop (Gulberg 2)** | Affordable seekh kebabs | ₹₵ |
| **Nasir Biryani** (near Gulberg) | Famous for spicy biryani | ₹₵ |
| **Bundoo's** | Old-school Pakistani cuisine | ₹₵ |
| **BBQ Tonight** | Buffet deals on certain days | ₹₵₵ |

### DHA (Affordable Picks)
| Place | What's Good | Price Range |
|-------|-------------|-------------|
| **Haveli DHA** | Affordable desi food with vibe | ₹₵ |
| **Subway** | Budget sandwiches & salads | ₹₵

OPENAI_API_KEY is not set, skipping trace export


Turn 3: # 🕐 Half-Day Plan: Gulberg & DHA

**Best route:** Gulberg → DHA (minimal backtracking)

---

## ⏰ 9:00 AM — Breakfast

**Khatri Food Corner** (Gulberg 2)

- Famous for *halwa puri*, *chana*, *naan chanay*
- Budget-friendly: ~₨ 200-300 per person
- 📍 Near Gulberg Main Boulevard

---

## ⏰ 9:45 AM — Quick Shopping

**Liberty Market** (5-min drive)

- Fashion, dupattas, accessories
- Bargain hard — start at 50% of asking price
- OR **Gulberg Underground** for shoes & clothes

---

## ⏰ 11:00 AM — Head to DHA

Short 10-min drive from Gulberg

---

## ⏰ 11:15 AM — Fresh Air Walk

**DHA Lake Park** (Phase 5)

- Peaceful walk/jog around the lake
- Free entry
- Great photo spots

---

## ⏰ 12:00 PM — Lunch

**DHA Food Street** (Phase 5 Commercial)

- Affordable grills, biryani, desi food stalls
- Budget: ~₨ 300-500 per person

**OR** nearby **Haveli DHA** for AC dining

---

## ⏰ 1:00 PM — Done! ✅

---

## 📍 Map Summary

```
[ Gulberg ]  → 10min  →  [ DHA ]
Breakfast      drive       L

OPENAI_API_KEY is not set, skipping trace export
